# Customer Churn Prediction & Retention Analytics (End-to-End Portfolio Study)
**Dataset**: IBM Telco Customer Churn (7,043 rows, 21 attributes)
**Objective**: Predict churn likelihood, uncover top risk drivers via TreeSHAP, and inform retention playbooks.


## 1. Environment Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.data_loader import load_raw_data, audit_data_quality
from src.preprocessing import clean_raw_data, split_data
from src.feature_engineering import FeaturePipeline
from src.train import train_and_tune_models

raw_df = load_raw_data()
audit = audit_data_quality(raw_df)
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

## 2. Data Cleaning & Preprocessing

In [2]:
df_clean = clean_raw_data(raw_df)
print(f"TotalCharges nulls after cleaning: {df_clean['TotalCharges'].isnull().sum()}")
print(f"Overall Churn Rate: {df_clean['Churn'].mean():.2%}")
train_df, test_df = split_data(df_clean, test_size=0.2, random_state=42)
print(f"Train set: {train_df.shape}, Test set: {test_df.shape}")

## 3. Feature Engineering & Scaling

In [3]:
pipeline = FeaturePipeline()
X_train = pipeline.fit_transform(train_df)
y_train = train_df["Churn"]
X_test = pipeline.transform(test_df)
y_test = test_df["Churn"]
print(f"Engineered feature matrix: {X_train.shape[1]} features")

## 4. Model Training & Comparison

In [4]:
import joblib
models, cv_results, best_model_name = train_and_tune_models(X_train, y_train)
cv_df = pd.DataFrame(cv_results).T
display(cv_df)

## 5. Model Evaluation & SHAP Interpretability

In [5]:
from src.evaluate import evaluate_models_on_test
from src.explain import generate_shap_explanations
comparison_df = evaluate_models_on_test()
shap_res = generate_shap_explanations()
print("Top 5 Churn Drivers:", shap_res["top_5_drivers"])